# Label Tool — Temporary YOLO Training
Paste the Config URL from Label Tool, then **Runtime → Run all**. Training files are temporary.

In [ ]:
CONFIG_URL=input('Paste training Config URL: ').strip()
assert CONFIG_URL.startswith('http')

In [ ]:
import json, os, shutil, urllib.request, zipfile
with urllib.request.urlopen(CONFIG_URL) as r: cfg=json.loads(r.read().decode())
shutil.rmtree('/content/label_dataset', ignore_errors=True)
urllib.request.urlretrieve(cfg['dataset_url'], '/content/dataset.zip')
with zipfile.ZipFile('/content/dataset.zip') as z: z.extractall('/content/label_dataset')
!pip install -q ultralytics pyyaml
import yaml
yaml_path='/content/label_dataset/data.yaml'
with open(yaml_path, 'r', encoding='utf-8') as f: dataset_cfg=yaml.safe_load(f)
dataset_cfg['path']='/content/label_dataset'
with open(yaml_path, 'w', encoding='utf-8') as f: yaml.safe_dump(dataset_cfg, f, sort_keys=False)
train_dir=os.path.join(dataset_cfg['path'], dataset_cfg.get('train', 'images/train'))
assert os.path.isdir(train_dir), f'Training images folder not found: {train_dir}'
print('Dataset ready:', train_dir, '-', len(os.listdir(train_dir)), 'images')
from ultralytics import YOLO
model=YOLO('yolo11n.pt')
result=model.train(data=yaml_path, epochs=int(cfg['epochs']), imgsz=int(cfg['image_size']), project='/content/runs', name='label_tool')
best_path=str(model.trainer.best)
assert os.path.isfile(best_path), f'Best model was not created: {best_path}'
print('Training complete. Best model:', best_path)
from google.colab import files
files.download(best_path)